# Mapper Runtime and Memory Benchmark

Compare `Swin3DLatentMapper(base)` and `NeighborGraphLatentMapper(base)` on the same `test.glb`-derived source/destination coordinates. The benchmark reports forward-only and forward+backward runtime and CUDA memory for float32 and bf16 autocast modes.

In [1]:
from __future__ import annotations

from contextlib import nullcontext
from dataclasses import replace
from pathlib import Path
import gc
import sys

import pandas as pd
import torch
import trimesh

ROOT = Path('/mnt/nvmefs/Projects/SymTRELLIS')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DEVICE = torch.device('cuda')
assert torch.cuda.is_available(), 'CUDA is required for this notebook'

from o_voxel.convert import intersect_occ
from symtrellis.mapper import (
    NeighborGraphLatentMapper,
    Swin3DLatentMapper,
    neighbor_graph_latent_mapper_config,
    swin_3d_latent_mapper_config,
)

G = 64
MESH_PATH = ROOT / 'notebooks/test.glb'
MESH_AABB = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]]
ROTATION_SEED = 20260628
BENCH_REPEATS = 10
WARMUP_REPEATS = 5

torch.manual_seed(20260628)
torch.cuda.manual_seed_all(20260628)

print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('device:', torch.cuda.get_device_name(DEVICE))
print('mesh:', MESH_PATH)
print('G:', G, 'BENCH_REPEATS:', BENCH_REPEATS, 'WARMUP_REPEATS:', WARMUP_REPEATS)

torch: 2.6.0+cu124 cuda: 12.4
device: NVIDIA GeForce RTX 4090
mesh: /mnt/nvmefs/Projects/SymTRELLIS/notebooks/test.glb
G: 64 BENCH_REPEATS: 10 WARMUP_REPEATS: 5


## Geometry Input

Source/KV coordinates are voxelized from the normalized mesh in the unrotated world grid. Q/destination coordinates are voxelized after a random rotation. The mapper transform maps rotated destination grid indices back into the unrotated source grid-index frame.

In [2]:
def cuda_timed(fn):
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    result = fn()
    end.record()
    torch.cuda.synchronize()
    return result, float(start.elapsed_time(end))


def scalar(x):
    return float(x.detach().item())


def grid_to_pos(grid):
    return (grid.to(torch.float32) + 0.5) / float(G) - 0.5


def random_rotation(seed, device):
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)
    q = torch.randn((4,), device=device, dtype=torch.float32, generator=gen)
    q = q / q.norm()
    w, x, y, z = q.unbind()
    return torch.stack([
        torch.stack([1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)]),
        torch.stack([2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)]),
        torch.stack([2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]),
    ])


def load_trimesh_mesh(path):
    loaded = trimesh.load(path, force='scene')
    if isinstance(loaded, trimesh.Scene):
        mesh = loaded.to_geometry()
        if not isinstance(mesh, trimesh.Trimesh) or len(mesh.vertices) == 0 or len(mesh.faces) == 0:
            raise ValueError(f'no triangle mesh found in {path}')
    elif isinstance(loaded, trimesh.Trimesh):
        mesh = loaded
    else:
        raise TypeError(f'unsupported mesh object: {type(loaded)}')
    mesh.remove_unreferenced_vertices()
    return mesh


@torch.no_grad()
def load_normalized_mesh(path):
    mesh = load_trimesh_mesh(path)
    vertices = torch.as_tensor(mesh.vertices, device=DEVICE, dtype=torch.float32).contiguous()
    faces = torch.as_tensor(mesh.faces, device=DEVICE, dtype=torch.long).contiguous()
    raw_min = vertices.min(dim=0).values
    raw_max = vertices.max(dim=0).values
    center = (raw_min + raw_max) * 0.5
    vertices = vertices - center
    raw_radius = vertices.norm(dim=1).max()
    scale = 0.5 / raw_radius
    vertices = (vertices * scale).contiguous()
    info = {
        'mesh_vertices': int(vertices.shape[0]),
        'mesh_faces': int(faces.shape[0]),
        'mesh_scale': scalar(scale),
        'mesh_raw_radius': scalar(raw_radius),
        'mesh_center_x': scalar(center[0]),
        'mesh_center_y': scalar(center[1]),
        'mesh_center_z': scalar(center[2]),
    }
    return vertices, faces, info


@torch.no_grad()
def voxelize_mapper_coords():
    vertices, faces, info = load_normalized_mesh(MESH_PATH)
    rot = random_rotation(ROTATION_SEED, DEVICE)

    src_grid = intersect_occ(vertices, faces, grid_size=G, aabb=MESH_AABB).to(torch.int32).contiguous()
    vertices_rot = (vertices @ rot).contiguous()
    dst_grid = intersect_occ(vertices_rot, faces, grid_size=G, aabb=MESH_AABB).to(torch.int32).contiguous()

    Nsrc = src_grid.shape[0]
    Ndst = dst_grid.shape[0]
    rel_src = torch.zeros((Nsrc, 1), device=DEVICE, dtype=torch.int32)
    rel_dst = torch.zeros((Ndst, 1), device=DEVICE, dtype=torch.int32)
    coords_src = torch.cat([rel_src, src_grid], dim=1).contiguous()
    coords_dst = torch.cat([rel_dst, dst_grid], dim=1).contiguous()

    # Mapper uses column-vector grid indices: pos_dst_in_src = O @ dst_grid + t.
    # This maps rotated destination/query voxel indices back to unrotated source/key indices.
    O_dst2src = rot.unsqueeze(0).contiguous()
    ones = torch.ones((3,), device=DEVICE, dtype=torch.float32)
    t_dst2src = (0.5 * float(G - 1) * (ones - rot @ ones)).unsqueeze(0).contiguous()
    s_dst2src = torch.tensor([0 if scalar(torch.linalg.det(rot)) >= 0 else 1], device=DEVICE, dtype=torch.long)

    dst_grid_float = coords_dst[:, 1:].to(torch.float32)
    mapped_by_mapper = (O_dst2src[0] @ dst_grid_float.T).T + t_dst2src[0]
    mapped_by_pos = (grid_to_pos(dst_grid_float) @ rot.T + 0.5) * float(G) - 0.5
    info.update({
        'Nsrc': int(Nsrc),
        'Ndst': int(Ndst),
        'rotation_det': scalar(torch.linalg.det(rot)),
        'transform_max_abs_error': scalar((mapped_by_mapper - mapped_by_pos).abs().max()),
    })
    return coords_src, coords_dst, O_dst2src, t_dst2src, s_dst2src, info


(coords_src, coords_dst, O_dst2src, t_dst2src, s_dst2src, geometry_info), voxelize_ms = cuda_timed(voxelize_mapper_coords)
geometry_info['voxelize_ms'] = voxelize_ms
display(pd.DataFrame([geometry_info]))
print('coords_src:', coords_src.shape, coords_src.dtype, coords_src.device)
print('coords_dst:', coords_dst.shape, coords_dst.dtype, coords_dst.device)
print('O:', O_dst2src.shape, 't:', t_dst2src.shape, 's:', s_dst2src.tolist())

,mesh_vertices,mesh_faces,mesh_scale,mesh_raw_radius,mesh_center_x,mesh_center_y,mesh_center_z,Nsrc,Ndst,rotation_det,transform_max_abs_error,voxelize_ms
0,268018,280333,0.823705,0.607013,0.00389,-0.000544,0.002542,16054,17010,1.0,0.000008,176.206329


coords_src: torch.Size([16054, 4]) torch.int32 cuda:0
coords_dst: torch.Size([17010, 4]) torch.int32 cuda:0
O: torch.Size([1, 3, 3]) t: torch.Size([1, 3]) s: [0]


## Mapper Builders

Float32 uses xFormers for the Swin mapper. bf16 uses CUDA autocast and flash-attn for the Swin mapper. The neighbor graph mapper uses CSR attention in both dtype modes and is benchmarked with `sort=True` and `sort=False`.

In [3]:
MAPPER_CASES = [
    {'case': 'swin3d', 'mapper': 'swin3d', 'sort': None},
    {'case': 'neighbor_graph_sort', 'mapper': 'neighbor_graph', 'sort': True},
    {'case': 'neighbor_graph_no_sort', 'mapper': 'neighbor_graph', 'sort': False},
]


def make_mapper(mapper_name, dtype_name):
    if mapper_name == 'swin3d':
        cfg = swin_3d_latent_mapper_config(scale='base')
        if dtype_name == 'bf16_autocast':
            cfg = replace(cfg, attn_backend='flash_attn')
        else:
            cfg = replace(cfg, attn_backend='xformers')
        model = Swin3DLatentMapper(cfg)
    elif mapper_name == 'neighbor_graph':
        cfg = neighbor_graph_latent_mapper_config(scale='base')
        model = NeighborGraphLatentMapper(cfg)
    else:
        raise ValueError(f'unknown mapper: {mapper_name}')
    return model.to(device=DEVICE, dtype=torch.float32).train(), cfg


def mapper_forward(model, mapper_name, sort):
    if mapper_name == 'neighbor_graph':
        return model(coords_src, coords_dst, O_dst2src, t_dst2src, s_dst2src, sort=sort)
    return model(coords_src, coords_dst, O_dst2src, t_dst2src, s_dst2src)


def param_stats(model):
    params = list(model.parameters())
    count = sum(p.numel() for p in params)
    bytes_ = sum(p.numel() * p.element_size() for p in params)
    return count, bytes_ / 1024**2


param_rows = []
for case in MAPPER_CASES:
    model, cfg = make_mapper(case['mapper'], 'float32')
    count, mb = param_stats(model)
    param_rows.append({
        'case': case['case'],
        'mapper': case['mapper'],
        'sort': case['sort'],
        'scale': 'base',
        'feat_dim': cfg.feat_dim,
        'num_heads': cfg.num_heads,
        'head_dim': cfg.feat_dim // cfg.num_heads,
        'depth': cfg.depth,
        'param_count': count,
        'param_mb_fp32': mb,
    })
    del model
    torch.cuda.empty_cache()

param_df = pd.DataFrame(param_rows)
display(param_df)

,case,mapper,sort,scale,feat_dim,num_heads,head_dim,depth,param_count,param_mb_fp32
0,swin3d,swin3d,None,base,192,6,32,8,13053264,49.794250
1,neighbor_graph_sort,neighbor_graph,True,base,192,6,32,8,12905424,49.230286
2,neighbor_graph_no_sort,neighbor_graph,False,base,192,6,32,8,12905424,49.230286


## Benchmark

Forward-only runs under `torch.no_grad()`. Forward+backward uses `loss = mean(coeff.s^2) + mean(coeff.w^2)` so gradients flow through mapper parameters. Memory deltas are measured relative to the state immediately before each measured run; total peaks are also reported.

In [4]:
def autocast_context(dtype_name):
    if dtype_name == 'bf16_autocast':
        return torch.autocast(device_type='cuda', dtype=torch.bfloat16)
    return nullcontext()


def coeff_loss(coeff):
    return coeff.s.float().square().mean() + coeff.w.float().square().mean()


def run_once(model, mapper_name, sort, dtype_name, mode, measured=True):
    coeff = None
    loss = None
    model.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    if measured:
        torch.cuda.reset_peak_memory_stats()
    base_alloc = torch.cuda.memory_allocated()
    base_reserved = torch.cuda.memory_reserved()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    try:
        start.record()
        if mode == 'forward':
            with torch.no_grad():
                with autocast_context(dtype_name):
                    coeff = mapper_forward(model, mapper_name, sort)
        elif mode == 'forward_backward':
            with autocast_context(dtype_name):
                coeff = mapper_forward(model, mapper_name, sort)
                loss = coeff_loss(coeff)
            loss.backward()
        else:
            raise ValueError(f'unknown mode: {mode}')
        end.record()
        torch.cuda.synchronize()
        record = {
            'status': 'ok',
            'ms': float(start.elapsed_time(end)),
            'alloc_delta_mb': (torch.cuda.memory_allocated() - base_alloc) / 1024**2,
            'reserved_delta_mb': (torch.cuda.memory_reserved() - base_reserved) / 1024**2,
            'peak_alloc_delta_mb': (torch.cuda.max_memory_allocated() - base_alloc) / 1024**2,
            'peak_reserved_delta_mb': (torch.cuda.max_memory_reserved() - base_reserved) / 1024**2,
            'peak_alloc_total_mb': torch.cuda.max_memory_allocated() / 1024**2,
            'peak_reserved_total_mb': torch.cuda.max_memory_reserved() / 1024**2,
        }
    except RuntimeError as err:
        torch.cuda.synchronize()
        message = str(err).split('\n')[0]
        record = {
            'status': 'oom' if 'out of memory' in message.lower() else 'error',
            'error': message,
            'ms': float('nan'),
            'alloc_delta_mb': float('nan'),
            'reserved_delta_mb': float('nan'),
            'peak_alloc_delta_mb': float('nan'),
            'peak_reserved_delta_mb': float('nan'),
            'peak_alloc_total_mb': torch.cuda.max_memory_allocated() / 1024**2,
            'peak_reserved_total_mb': torch.cuda.max_memory_reserved() / 1024**2,
        }
    finally:
        coeff = None
        loss = None
        model.zero_grad(set_to_none=True)
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    return record


def summarize_records(records):
    ok = [r for r in records if r['status'] == 'ok']
    if not ok:
        return {
            'status': records[-1]['status'],
            'error': records[-1].get('error', ''),
            'runtime_ms_mean': float('nan'),
            'runtime_ms_min': float('nan'),
            'alloc_delta_mb_mean': float('nan'),
            'reserved_delta_mb_mean': float('nan'),
            'peak_alloc_delta_mb_mean': float('nan'),
            'peak_reserved_delta_mb_mean': float('nan'),
            'peak_alloc_total_mb_mean': float('nan'),
            'peak_reserved_total_mb_mean': float('nan'),
        }
    return {
        'status': 'ok',
        'error': '',
        'runtime_ms_mean': sum(r['ms'] for r in ok) / len(ok),
        'runtime_ms_min': min(r['ms'] for r in ok),
        'alloc_delta_mb_mean': sum(r['alloc_delta_mb'] for r in ok) / len(ok),
        'reserved_delta_mb_mean': sum(r['reserved_delta_mb'] for r in ok) / len(ok),
        'peak_alloc_delta_mb_mean': sum(r['peak_alloc_delta_mb'] for r in ok) / len(ok),
        'peak_reserved_delta_mb_mean': sum(r['peak_reserved_delta_mb'] for r in ok) / len(ok),
        'peak_alloc_total_mb_mean': sum(r['peak_alloc_total_mb'] for r in ok) / len(ok),
        'peak_reserved_total_mb_mean': sum(r['peak_reserved_total_mb'] for r in ok) / len(ok),
    }


bench_rows = []
for case in MAPPER_CASES:
    case_name = case['case']
    mapper_name = case['mapper']
    sort = case['sort']
    for dtype_name in ['float32', 'bf16_autocast']:
        print('build', case_name, dtype_name)
        model, cfg = make_mapper(mapper_name, dtype_name)
        param_count, param_mb = param_stats(model)
        backend = getattr(cfg, 'attn_backend', 'csr')
        for mode in ['forward', 'forward_backward']:
            print('bench', case_name, dtype_name, mode, 'backend', backend, 'sort', sort)
            warmup = [run_once(model, mapper_name, sort, dtype_name, mode, measured=False) for _ in range(WARMUP_REPEATS)]
            if warmup and warmup[-1]['status'] != 'ok':
                summary = summarize_records(warmup)
            else:
                records = [run_once(model, mapper_name, sort, dtype_name, mode, measured=True) for _ in range(BENCH_REPEATS)]
                summary = summarize_records(records)
            bench_rows.append({
                'case': case_name,
                'mapper': mapper_name,
                'dtype_mode': dtype_name,
                'mode': mode,
                'attn_backend': backend,
                'sort': sort,
                'scale': 'base',
                'feat_dim': cfg.feat_dim,
                'num_heads': cfg.num_heads,
                'head_dim': cfg.feat_dim // cfg.num_heads,
                'depth': cfg.depth,
                'param_count': param_count,
                'param_mb_fp32': param_mb,
                'Nsrc': int(coords_src.shape[0]),
                'Ndst': int(coords_dst.shape[0]),
                'repeats': BENCH_REPEATS,
                **summary,
            })
        del model
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

bench_df = pd.DataFrame(bench_rows)
display(bench_df)

build swin3d float32
bench swin3d float32 forward backend xformers sort None
bench swin3d float32 forward_backward backend xformers sort None
build swin3d bf16_autocast
bench swin3d bf16_autocast forward backend flash_attn sort None
bench swin3d bf16_autocast forward_backward backend flash_attn sort None
build neighbor_graph_sort float32
bench neighbor_graph_sort float32 forward backend csr sort True
bench neighbor_graph_sort float32 forward_backward backend csr sort True
build neighbor_graph_sort bf16_autocast
bench neighbor_graph_sort bf16_autocast forward backend csr sort True
bench neighbor_graph_sort bf16_autocast forward_backward backend csr sort True
build neighbor_graph_no_sort float32
bench neighbor_graph_no_sort float32 forward backend csr sort False
bench neighbor_graph_no_sort float32 forward_backward backend csr sort False
build neighbor_graph_no_sort bf16_autocast
bench neighbor_graph_no_sort bf16_autocast forward backend csr sort False
bench neighbor_graph_no_sort bf16_a

,case,mapper,dtype_mode,mode,attn_backend,sort,scale,feat_dim,num_heads,head_dim,...,status,error,runtime_ms_mean,runtime_ms_min,alloc_delta_mb_mean,reserved_delta_mb_mean,peak_alloc_delta_mb_mean,peak_reserved_delta_mb_mean,peak_alloc_total_mb_mean,peak_reserved_total_mb_mean
0,swin3d,swin3d,float32,forward,xformers,None,base,192,6,32,...,ok,,80.047799,78.525436,104.523926,3916.0,3548.129395,3916.0,3607.235352,3988.0
1,swin3d,swin3d,float32,forward_backward,xformers,None,base,192,6,32,...,ok,,762.347888,741.696533,162.093750,14958.0,12724.594727,14958.0,12791.941895,15050.0
2,swin3d,swin3d,bf16_autocast,forward,flash_attn,None,base,192,6,32,...,ok,,58.132778,57.331936,59.317383,3462.0,3187.870117,3462.0,3255.101074,3534.0
3,swin3d,swin3d,bf16_autocast,forward_backward,flash_attn,None,base,192,6,32,...,ok,,586.536041,575.303650,110.519043,10696.0,9111.286133,10696.0,9178.517090,10768.0
4,neighbor_graph_sort,neighbor_graph,float32,forward,csr,True,base,192,6,32,...,ok,,75.880493,72.620033,104.746582,4002.0,3602.985352,4002.0,3669.652344,4074.0
5,neighbor_graph_sort,neighbor_graph,float32,forward_backward,csr,True,base,192,6,32,...,ok,,674.630603,642.539490,161.497559,13374.0,11136.303711,13374.0,11202.970703,13446.0
6,neighbor_graph_sort,neighbor_graph,bf16_autocast,forward,csr,True,base,192,6,32,...,ok,,62.057351,55.470142,58.529785,3518.0,3244.041992,3518.0,3310.708984,3590.0
7,neighbor_graph_sort,neighbor_graph,bf16_autocast,forward_backward,csr,True,base,192,6,32,...,ok,,616.565283,587.359253,110.181152,10142.0,8456.529785,10142.0,8523.196777,10214.0
8,neighbor_graph_no_sort,neighbor_graph,float32,forward,csr,False,base,192,6,32,...,ok,,71.143118,69.147652,104.998535,3974.0,3577.990723,3974.0,3644.657715,4046.0
9,neighbor_graph_no_sort,neighbor_graph,float32,forward_backward,csr,False,base,192,6,32,...,ok,,670.225586,639.692078,161.274902,13334.0,11106.844727,13334.0,11173.511719,13406.0


## Summary Views

In [5]:
display(bench_df[[
    'case', 'mapper', 'sort', 'dtype_mode', 'mode', 'attn_backend', 'status',
    'runtime_ms_mean', 'runtime_ms_min',
    'alloc_delta_mb_mean', 'peak_alloc_delta_mb_mean',
    'reserved_delta_mb_mean', 'peak_reserved_delta_mb_mean',
    'peak_alloc_total_mb_mean', 'peak_reserved_total_mb_mean',
    'param_count', 'param_mb_fp32', 'Nsrc', 'Ndst', 'error',
]])

ok_df = bench_df[bench_df['status'] == 'ok'].copy()
if not ok_df.empty:
    pivot = ok_df.pivot_table(
        index=['dtype_mode', 'mode'],
        columns='case',
        values=['runtime_ms_mean', 'peak_alloc_delta_mb_mean', 'peak_reserved_delta_mb_mean'],
        aggfunc='first',
    )
    display(pivot)
else:
    print('No successful benchmark rows to summarize.')

,case,mapper,sort,dtype_mode,mode,attn_backend,status,runtime_ms_mean,runtime_ms_min,alloc_delta_mb_mean,peak_alloc_delta_mb_mean,reserved_delta_mb_mean,peak_reserved_delta_mb_mean,peak_alloc_total_mb_mean,peak_reserved_total_mb_mean,param_count,param_mb_fp32,Nsrc,Ndst,error
0,swin3d,swin3d,None,float32,forward,xformers,ok,80.047799,78.525436,104.523926,3548.129395,3916.0,3916.0,3607.235352,3988.0,13053264,49.794250,16054,17010,
1,swin3d,swin3d,None,float32,forward_backward,xformers,ok,762.347888,741.696533,162.093750,12724.594727,14958.0,14958.0,12791.941895,15050.0,13053264,49.794250,16054,17010,
2,swin3d,swin3d,None,bf16_autocast,forward,flash_attn,ok,58.132778,57.331936,59.317383,3187.870117,3462.0,3462.0,3255.101074,3534.0,13053264,49.794250,16054,17010,
3,swin3d,swin3d,None,bf16_autocast,forward_backward,flash_attn,ok,586.536041,575.303650,110.519043,9111.286133,10696.0,10696.0,9178.517090,10768.0,13053264,49.794250,16054,17010,
4,neighbor_graph_sort,neighbor_graph,True,float32,forward,csr,ok,75.880493,72.620033,104.746582,3602.985352,4002.0,4002.0,3669.652344,4074.0,12905424,49.230286,16054,17010,
5,neighbor_graph_sort,neighbor_graph,True,float32,forward_backward,csr,ok,674.630603,642.539490,161.497559,11136.303711,13374.0,13374.0,11202.970703,13446.0,12905424,49.230286,16054,17010,
6,neighbor_graph_sort,neighbor_graph,True,bf16_autocast,forward,csr,ok,62.057351,55.470142,58.529785,3244.041992,3518.0,3518.0,3310.708984,3590.0,12905424,49.230286,16054,17010,
7,neighbor_graph_sort,neighbor_graph,True,bf16_autocast,forward_backward,csr,ok,616.565283,587.359253,110.181152,8456.529785,10142.0,10142.0,8523.196777,10214.0,12905424,49.230286,16054,17010,
8,neighbor_graph_no_sort,neighbor_graph,False,float32,forward,csr,ok,71.143118,69.147652,104.998535,3577.990723,3974.0,3974.0,3644.657715,4046.0,12905424,49.230286,16054,17010,
9,neighbor_graph_no_sort,neighbor_graph,False,float32,forward_backward,csr,ok,670.225586,639.692078,161.274902,11106.844727,13334.0,13334.0,11173.511719,13406.0,12905424,49.230286,16054,17010,


peak_alloc_delta_mb_mean                      \
case                             neighbor_graph_no_sort neighbor_graph_sort   
dtype_mode    mode                                                            
bf16_autocast forward                       3218.953125         3244.041992   
              forward_backward              8430.623535         8456.529785   
float32       forward                       3577.990723         3602.985352   
              forward_backward             11106.844727        11136.303711   

                                             peak_reserved_delta_mb_mean  \
case                                  swin3d      neighbor_graph_no_sort   
dtype_mode    mode                                                         
bf16_autocast forward            3187.870117                      3522.0   
              forward_backward   9111.286133                     10118.0   
float32       forward            3548.129395                      3974.0   
              forward_backward  12724.594727                     13334.0   

                                                             \
case                           neighbor_graph_sort   swin3d   
dtype_mode    mode                                            
bf16_autocast forward                       3518.0   3462.0   
              forward_backward             10142.0  10696.0   
float32       forward                       4002.0   3916.0   
              forward_backward             13374.0  14958.0   

                                      runtime_ms_mean                      \
case                           neighbor_graph_no_sort neighbor_graph_sort   
dtype_mode    mode                                                          
bf16_autocast forward                       59.213107           62.057351   
              forward_backward             637.736353          616.565283   
float32       forward                       71.143118           75.880493   
              forward_backward             670.225586          674.630603   

                                            
case                                swin3d  
dtype_mode    mode                          
bf16_autocast forward            58.132778  
              forward_backward  586.536041  
float32       forward            80.047799  
              forward_backward  762.347888